In [14]:
import os
import pandas as pd
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Reshape # MODIFIED: Import Reshape
from collections import defaultdict, deque

# --- 1. Configuration & Setup ---
CSV_PATH = "D:/Project_P1/results/video_analysis/test_video_detections_with_ids.csv"
VIDEO_PATH = "D:/Project_P1/data/valid/test_video.mp4" 
OUTPUT_VIDEO_PATH = "D:/Project_P1/results/video_analysis/test1_with_5frame_predictions.mp4"

# MODIFIED: Define sequence lengths for input and output
SEQUENCE_LENGTH = 15      # Use 15 past frames to predict
PREDICTION_STEPS = 5      # Predict 5 frames into the future
EPOCHS = 75               # Increased epochs slightly for the more complex task

# --- 2. Data Loading and Preprocessing for Training ---
print("--- Step 1: Loading and Preparing Data for Multi-Step Prediction ---")
df = pd.read_csv(CSV_PATH)

cap_temp = cv2.VideoCapture(VIDEO_PATH)
frame_width = int(cap_temp.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap_temp.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap_temp.release()
print(f"Video dimensions: {frame_width}x{frame_height}")

df['center_x_norm'] = df['center_x'] / frame_width
df['center_y_norm'] = df['center_y'] / frame_height

# MODIFIED: Create sequences for many-to-many prediction
all_sequences_X, all_sequences_y = [], []
for track_id in df['track_id'].unique():
    bird_path = df[df['track_id'] == track_id][['center_x_norm', 'center_y_norm']].values
    
    # Adjust loop range to account for the 5 future steps
    if len(bird_path) > SEQUENCE_LENGTH + PREDICTION_STEPS:
        for i in range(len(bird_path) - SEQUENCE_LENGTH - PREDICTION_STEPS):
            all_sequences_X.append(bird_path[i:i + SEQUENCE_LENGTH])
            # The target 'y' is now the next 5 points
            all_sequences_y.append(bird_path[i + SEQUENCE_LENGTH : i + SEQUENCE_LENGTH + PREDICTION_STEPS])

X_train = np.array(all_sequences_X)
y_train = np.array(all_sequences_y)
print(f"Created {X_train.shape[0]} sequences for training.")

# --- 3. Build and Train the LSTM Model ---
print("\n--- Step 2: Building and Training the Many-to-Many LSTM Model ---")
model = Sequential([
    LSTM(units=100, activation='relu', input_shape=(SEQUENCE_LENGTH, 2)),
    Dense(units=50, activation='relu'),
    # MODIFIED: Output layer now has 10 neurons (5 steps * 2 coords)
    Dense(units=PREDICTION_STEPS * 2),
    # MODIFIED: Reshape the output to (5, 2) for clarity
    Reshape((PREDICTION_STEPS, 2))
])
model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()
model.fit(X_train, y_train, epochs=EPOCHS, batch_size=32, verbose=1)
print("✅ Model training complete.")

# --- 4. Inference and Visualization on Video ---
print("\n--- Step 3: Running Inference and Visualizing Future Path ---")
cap = cv2.VideoCapture(VIDEO_PATH)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, cap.get(cv2.CAP_PROP_FPS), (frame_width, frame_height))

detections_by_frame = {frame: group for frame, group in df.groupby('frame')}
track_histories = defaultdict(lambda: deque(maxlen=SEQUENCE_LENGTH))

frame_number = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break

    if frame_number in detections_by_frame:
        current_detections = detections_by_frame[frame_number]
        for _, detection in current_detections.iterrows():
            track_id = int(detection['track_id'])
            center_x, center_y = int(detection['center_x']), int(detection['center_y'])
            track_histories[track_id].append((center_x, center_y))

            if len(track_histories[track_id]) == SEQUENCE_LENGTH:
                history_np = np.array(track_histories[track_id])
                normalized_history = history_np / [frame_width, frame_height]
                input_seq = normalized_history.reshape(1, SEQUENCE_LENGTH, 2)
                
                # Model now predicts a sequence of 5 points
                predicted_normalized = model.predict(input_seq)
                
                # Denormalize the entire predicted path
                predicted_path = (predicted_normalized[0] * [frame_width, frame_height]).astype(int)
                
                # MODIFIED: Draw the predicted path as a polyline
                # First, connect the current point to the first predicted point
                full_path_for_drawing = np.vstack([history_np[-1], predicted_path])
                cv2.polylines(frame, [full_path_for_drawing], isClosed=False, color=(0, 255, 0), thickness=2)
    
    # Draw historical paths (in blue)
    for track_id, history in track_histories.items():
        if len(history) > 1:
            points = np.array(history, dtype=np.int32).reshape((-1, 1, 2))
            cv2.polylines(frame, [points], isClosed=False, color=(255, 0, 0), thickness=2)

    out.write(frame)
    if frame_number % 100 == 0: print(f"  Processed frame {frame_number}...")
    frame_number += 1

# --- 5. Finalize ---
cap.release()
out.release()
cv2.destroyAllWindows()
print(f"\n✅ Success! Video with 5-frame future path predictions saved to: {OUTPUT_VIDEO_PATH}")

--- Step 1: Loading and Preparing Data for Multi-Step Prediction ---
Video dimensions: 898x506
Created 370 sequences for training.

--- Step 2: Building and Training the Many-to-Many LSTM Model ---


c:\Users\visha\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 100)            │        41,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 50)             │         5,050 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │           510 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 5, 2)           │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 46,760 (182.66 KB)

 Trainable params: 46,760 (182.66 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/75


KeyboardInterrupt: 

In [ ]:
import numpy as np
import pandas as pd

# --- Configuration ---
NUM_TRAJECTORIES = 500  # Total number of unique flight paths to generate
MIN_LEN = 50            # Minimum frames in a path
MAX_LEN = 200           # Maximum frames in a path
WIDTH = 1920            # Frame width for coordinate space
HEIGHT = 1080           # Frame height for coordinate space

def generate_linear_path(length, start_pos, velocity):
    """Generates a straight path with constant velocity."""
    path = np.zeros((length, 2))
    for i in range(length):
        path[i] = start_pos + velocity * i
    return path

def generate_parabolic_path(length, start_pos, velocity, acceleration):
    """Generates a curved path with constant acceleration."""
    path = np.zeros((length, 2))
    for i in range(length):
        path[i] = start_pos + velocity * i + 0.5 * acceleration * i**2
    return path

def generate_sinusoidal_path(length, start_pos, velocity, amplitude, frequency):
    """Generates a path with a wave-like pattern (flapping)."""
    path = np.zeros((length, 2))
    for i in range(length):
        path[i] = start_pos + velocity * i
        # Add a sine wave perpendicular to the direction of motion
        perp_direction = np.array([-velocity[1], velocity[0]]) / np.linalg.norm(velocity)
        path[i] += perp_direction * amplitude * np.sin(frequency * i)
    return path

# --- Main Generation Script ---
print(f"Generating {NUM_TRAJECTORIES} synthetic bird trajectories...")
all_paths = []
track_id_counter = 0

for i in range(NUM_TRAJECTORIES):
    length = np.random.randint(MIN_LEN, MAX_LEN)
    start_pos = np.array([np.random.uniform(0.1*WIDTH, 0.9*WIDTH), 
                          np.random.uniform(0.1*HEIGHT, 0.9*HEIGHT)])
    
    # Randomly choose a path type
    path_type = np.random.choice(['linear', 'parabolic', 'sinusoidal'])
    
    if path_type == 'linear':
        velocity = np.random.uniform(-5, 5, 2)
        path = generate_linear_path(length, start_pos, velocity)
        
    elif path_type == 'parabolic':
        velocity = np.random.uniform(-3, 3, 2)
        acceleration = np.random.uniform(-0.1, 0.1, 2)
        path = generate_parabolic_path(length, start_pos, velocity, acceleration)
        
    elif path_type == 'sinusoidal':
        velocity = np.random.uniform(-4, 4, 2)
        amplitude = np.random.uniform(5, 20)
        frequency = np.random.uniform(0.1, 0.5)
        path = generate_sinusoidal_path(length, start_pos, velocity, amplitude, frequency)
        
    # Add some noise to make it more realistic
    path += np.random.normal(0, 1.5, path.shape)
    
    # Filter out paths that go off-screen
    on_screen = (path[:, 0] > 0) & (path[:, 0] < WIDTH) & (path[:, 1] > 0) & (path[:, 1] < HEIGHT)
    path = path[on_screen]
    
    if len(path) > MIN_LEN:
        # Create a DataFrame for this single path
        df_path = pd.DataFrame(path, columns=['x', 'y'])
        df_path['track_id'] = track_id_counter
        df_path['frame_id'] = range(len(path))
        all_paths.append(df_path)
        track_id_counter += 1

# Combine all generated paths into a single DataFrame
final_dataset = pd.concat(all_paths, ignore_index=True)

# Save to CSV
output_path = "D:/Project_P1/data/synthetic_bird_trajectories.csv"
final_dataset.to_csv(output_path, index=False)

print(f"\n✅ Success! Generated a dataset with {final_dataset['track_id'].nunique()} unique trajectories.")
print(f"Dataset saved to: {output_path}")

Generating 500 synthetic bird trajectories...

✅ Success! Generated a dataset with 480 unique trajectories.
Dataset saved to: D:/Project_P1/data/synthetic_bird_trajectories.csv


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Reshape
from sklearn.model_selection import train_test_split
import os

# --- 1. Configuration ---
# Path to the dataset we generated in the previous step
DATASET_PATH = "D:/Project_P1/data/synthetic_bird_trajectories.csv"
SAVED_MODEL_PATH = "D:/Project_P1/dev/universal_bird_predictor.h5"

# The dimensions used when generating the synthetic data
# This is crucial for correct normalization.
WIDTH = 1920
HEIGHT = 1080

# Model parameters (must match the desired input/output)
SEQUENCE_LENGTH = 15      # Number of past frames to use as input
PREDICTION_STEPS = 5      # Number of future frames to predict
EPOCHS = 100
BATCH_SIZE = 64

# --- 2. Load and Preprocess the Data ---
print("--- Loading and Preprocessing Data ---")
df = pd.read_csv(DATASET_PATH)

# Normalize the coordinates to a [0, 1] range
df['x_norm'] = df['x'] / WIDTH
df['y_norm'] = df['y'] / HEIGHT

# Prepare sequences for the LSTM model
all_sequences_X, all_sequences_y = [], []
# Group data by each unique trajectory
for track_id in df['track_id'].unique():
    # Get the normalized path for the current bird
    bird_path = df[df['track_id'] == track_id][['x_norm', 'y_norm']].values
    
    # Create sliding windows of input and output sequences
    if len(bird_path) > SEQUENCE_LENGTH + PREDICTION_STEPS:
        for i in range(len(bird_path) - SEQUENCE_LENGTH - PREDICTION_STEPS):
            # Input: A sequence of 15 past points
            all_sequences_X.append(bird_path[i:i + SEQUENCE_LENGTH])
            # Output: The corresponding 5 future points
            all_sequences_y.append(bird_path[i + SEQUENCE_LENGTH : i + SEQUENCE_LENGTH + PREDICTION_STEPS])

# Convert lists to numpy arrays
X = np.array(all_sequences_X)
y = np.array(all_sequences_y)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Data prepared successfully.")
print(f"Training sequences shape: {X_train.shape}")
print(f"Training targets shape: {y_train.shape}")
print(f"Validation sequences shape: {X_val.shape}")
print(f"Validation targets shape: {y_val.shape}")


# --- 3. Build the LSTM Model ---
print("\n--- Building the LSTM Model ---")
model = Sequential([
    LSTM(units=128, activation='relu', input_shape=(SEQUENCE_LENGTH, 2), return_sequences=True),
    LSTM(units=64, activation='relu'),
    Dense(units=32, activation='relu'),
    # Output layer needs to predict 5 (x,y) pairs, so 10 neurons total
    Dense(units=PREDICTION_STEPS * 2),
    # Reshape the flat output into 5 pairs of (x,y) coordinates
    Reshape((PREDICTION_STEPS, 2))
])


# --- 4. Compile and Train the Model ---
print("\n--- Compiling and Training the Model ---")
model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

# Train the model, using the validation set to monitor performance
history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    verbose=1
)

print("\n✅ Model training complete.")


# --- 5. Save the Final Model ---
print(f"\n--- Saving the trained model to {SAVED_MODEL_PATH} ---")
# Ensure the directory exists
os.makedirs(os.path.dirname(SAVED_MODEL_PATH), exist_ok=True)
model.save(SAVED_MODEL_PATH)
print("✅ Model saved successfully.")

--- Loading and Preprocessing Data ---
Data prepared successfully.
Training sequences shape: (34707, 15, 2)
Training targets shape: (34707, 5, 2)
Validation sequences shape: (8677, 15, 2)
Validation targets shape: (8677, 5, 2)

--- Building the LSTM Model ---

--- Compiling and Training the Model ---


c:\Users\visha\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_5 (LSTM)                   │ (None, 15, 128)        │        67,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 10)             │           330 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape_2 (Reshape)             │ (None, 5, 2)           │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 118,890 (464.41 KB)

 Trainable params: 118,890 (464.41 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
543/543 ━━━━━━━━━━━━━━━━━━━━ 11s 16ms/step - loss: 0.0424 - val_loss: 4.7737e-04
Epoch 2/100
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - loss: 4.2336e-04 - val_loss: 3.4081e-04
Epoch 3/100
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - loss: 2.8209e-04 - val_loss: 1.5267e-04
Epoch 4/100
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - loss: 2.0264e-04 - val_loss: 2.0952e-04
Epoch 5/100
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 17ms/step - loss: 1.5647e-04 - val_loss: 9.5425e-05
Epoch 6/100
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - loss: 1.1907e-04 - val_loss: 4.1766e-04
Epoch 7/100
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 16ms/step - loss: 1.1107e-04 - val_loss: 8.4509e-05
Epoch 8/100
543/543 ━━━━━━━━━━━━━━━━━━━━ 10s 18ms/step - loss: 8.5095e-05 - val_loss: 1.1887e-04
Epoch 9/100
543/543 ━━━━━━━━━━━━━━━━━━━━ 13s 23ms/step - loss: 8.5678e-05 - val_loss: 1.0133e-04
Epoch 10/100
543/543 ━━━━━━━━━━━━━━━━━━━━ 12s 23ms/step - loss: 8.1463e-05 - val_loss: 8.7971e-05
Epoch 11/100
543/543 ━━━━━━━━━━━━━━━━━━


✅ Model training complete.

--- Saving the trained model to D:/Project_P1/dev/universal_bird_predictor.h5 ---
✅ Model saved successfully.


In [ ]:
import pandas as pd
import numpy as np
import cv2
import tensorflow as tf
from collections import defaultdict, deque
import os

# --- 1. Configuration ---
TFLITE_MODEL_PATH = "D:/Project_P1/results/video_analysis/model.tflite"
DETECTIONS_CSV_PATH = "D:/Project_P1/results/video_analysis/test_video_detections_with_ids.csv"
VIDEO_PATH = "D:/Project_P1/data/valid/test_video.mp4"
OUTPUT_VIDEO_PATH = "D:/Project_P1/results/video_analysis/final_prediction_output.mp4"

SEQUENCE_LENGTH = 15
PREDICTION_STEPS = 5
ARROW_EXTENSION = 5  # Pixels to extend the arrow for better visibility

# --- 2. Load the Pre-trained TFLite Model ---
print("--- Loading TFLite model ---")
interpreter = tf.lite.Interpreter(model_path=TFLITE_MODEL_PATH)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print("✅ Model loaded successfully.")

# --- 3. Load Video and Detections ---
print("--- Loading video and detection data ---")
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    print(f"Error: Could not open video file at {VIDEO_PATH}")
    exit()

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))
print(f"Video dimensions: {frame_width}x{frame_height}")

df_detections = pd.read_csv(DETECTIONS_CSV_PATH)
detections_by_frame = {frame: group for frame, group in df_detections.groupby('frame')}

# --- 4. Process Video and Predict Frame-by-Frame ---
print("--- Starting prediction and visualization process ---")
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (frame_width, frame_height))
track_histories = defaultdict(lambda: deque(maxlen=SEQUENCE_LENGTH))

# Keep track of which track_ids were seen in the current frame
current_frame_track_ids = set()

def draw_arrow_with_extension(img, start_point, end_point, color, thickness):
    """Draw an arrow with extended length for better visibility"""
    # Calculate direction vector
    dx = end_point[0] - start_point[0]
    dy = end_point[1] - start_point[1]
    
    # Calculate length of the vector
    length = np.sqrt(dx**2 + dy**2)
    
    if length > 0:  # Avoid division by zero
        # Normalize the direction vector
        dx /= length
        dy /= length
        
        # Extend the end point by ARROW_EXTENSION pixels
        extended_end = (
            int(end_point[0] + dx * ARROW_EXTENSION),
            int(end_point[1] + dy * ARROW_EXTENSION)
        )
        
        # Draw the arrow
        cv2.arrowedLine(img, start_point, extended_end, color, thickness, tipLength=0.3)

frame_number = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # Reset the set for current frame
    current_frame_track_ids.clear()
    
    if frame_number in detections_by_frame:
        current_detections = detections_by_frame[frame_number]
        
        for _, detection in current_detections.iterrows():
            track_id = int(detection['track_id'])
            center_x, center_y = int(detection['center_x']), int(detection['center_y'])
            track_histories[track_id].append((center_x, center_y))
            current_frame_track_ids.add(track_id)  # Mark this track_id as present

            if len(track_histories[track_id]) == SEQUENCE_LENGTH:
                history_np = np.array(track_histories[track_id])
                
                normalized_history = history_np / [frame_width, frame_height]
                
                input_tensor = np.expand_dims(normalized_history, axis=0).astype(np.float32)
                
                interpreter.set_tensor(input_details[0]['index'], input_tensor)
                interpreter.invoke()
                predicted_normalized = interpreter.get_tensor(output_details[0]['index'])
                
                predicted_path = (predicted_normalized[0] * [frame_width, frame_height]).astype(int)
                
                # Draw the predicted path with arrows for better visibility
                start_point = tuple(history_np[-1])
                for i in range(len(predicted_path) - 1):
                    draw_arrow_with_extension(
                        frame, 
                        tuple(predicted_path[i]), 
                        tuple(predicted_path[i+1]), 
                        (0, 255, 0),  # Green color for predictions
                        2  # Thickness
                    )

    # Remove track histories for birds that are no longer detected
    tracks_to_remove = []
    for track_id in track_histories:
        if track_id not in current_frame_track_ids:
            tracks_to_remove.append(track_id)
    
    for track_id in tracks_to_remove:
        del track_histories[track_id]

    # Draw the historical paths for all currently tracked birds (in blue)
    for track_id, history in track_histories.items():
        if len(history) > 1:
            points = np.array(history, dtype=np.int32).reshape((-1, 1, 2))
            cv2.polylines(frame, [points], isClosed=False, color=(255, 0, 0), thickness=2)

    # Write the annotated frame to the output file
    out.write(frame)
    if frame_number > 0 and frame_number % 100 == 0:
        print(f"  Processed frame {frame_number}...")
    frame_number += 1

# --- 5. Finalize ---
cap.release()
out.release()
cv2.destroyAllWindows()
print(f"\n✅ Success! Video with predictions saved to: {OUTPUT_VIDEO_PATH}")

--- Loading TFLite model ---
✅ Model loaded successfully.
--- Loading video and detection data ---
Video dimensions: 898x506
--- Starting prediction and visualization process ---
  Processed frame 100...
  Processed frame 200...
  Processed frame 300...

✅ Success! Video with predictions saved to: D:/Project_P1/results/video_analysis/final_prediction_output.mp4


In [22]:
import os
import cv2
import pandas as pd
import numpy as np
import tensorflow as tf
from collections import defaultdict
import random
import time
import math

# ---------------- Configuration ----------------
DETECTIONS_CSV_PATH = "D:/Project_P1/results/video_analysis/test1_detections_with_ids.csv"
VIDEO_PATH = "D:/Project_P1/data/valid/test1.mp4"
TFLITE_MODEL_PATH = "D:/Project_P1/results/video_analysis/model.tflite"
OUTPUT_VIDEO_PATH = "D:/Project_P1/results/video_analysis/video_with_multi_bird_predictions_optimized.mp4"

SEQUENCE_LENGTH = 15
PREDICTION_HORIZON = 5

# Run prediction every N frames per track (1 = every frame)
PRED_EVERY_N_FRAMES = 2

# Performance thresholds
MAX_PROCESSING_TIME_MS = 33
PREDICTION_TIME_WARNING_THRESHOLD_MS = 10

# Visualization params (reduced cost)
FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.45
PATH_THICKNESS = 1
BOX_THICKNESS = 1
PREDICTION_THICKNESS = 1
DOT_RADIUS = 3
TEXT_THICKNESS = 1

# Misc
COLOR_PALETTE_SIZE = 50
AVG_PRED_STATS_WINDOW = 30  # compute avg pred time every N predictions for display

# ---------------- Utility ----------------
def generate_colors(num_colors):
    colors = []
    rng = random.Random(42)
    for i in range(num_colors):
        r = rng.randint(60, 255)
        g = rng.randint(60, 255)
        b = rng.randint(60, 255)
        colors.append((int(b), int(g), int(r)))
    return colors

# ---------------- Flight Pattern Analysis (unchanged) ----------------
def analyze_multi_bird_patterns(df):
    patterns = {}
    metrics = {}
    for track_id, group in df.groupby('track_id'):
        if len(group) < 5:
            continue
        seq_data = group.sort_values('frame').copy()
        x = seq_data['center_x'].values
        y = seq_data['center_y'].values
        dx, dy = np.gradient(x), np.gradient(y)
        velocity = np.sqrt(dx**2 + dy**2)
        ddx, ddy = np.gradient(dx), np.gradient(dy)
        acceleration = np.sqrt(ddx**2 + ddy**2)
        dddx, dddy = np.gradient(ddx), np.gradient(ddy)
        jerk = np.sqrt(dddx**2 + dddy**2)
        directions = np.arctan2(dy, dx)
        turning_angles = np.abs(np.diff(directions))
        turning_angles = np.append(turning_angles, 0)
        avg_vel, std_vel = np.mean(velocity), np.std(velocity)
        avg_acc, std_acc = np.mean(acceleration), np.std(acceleration)
        avg_jerk = np.mean(jerk)
        avg_turn = np.mean(turning_angles)
        pattern_type = "Unknown"
        if std_vel < 5 and avg_acc < 2 and avg_turn < 0.2:
            pattern_type = "Steady flight"
        elif avg_acc > 5 or avg_turn > 0.5:
            pattern_type = "Maneuvering"
        elif std_vel > 10 or avg_jerk > 3:
            pattern_type = "Erratic flight"
        patterns[track_id] = pattern_type
        metrics[track_id] = {
            'avg_velocity': avg_vel,
            'std_velocity': std_vel,
            'avg_acceleration': avg_acc,
            'std_acceleration': std_acc,
            'avg_jerk': avg_jerk,
            'avg_turning_angle': avg_turn,
            'frames_tracked': len(group)
        }
    return patterns, metrics

# ---------------- Per-track ring buffer ----------------
class TrackBuffer:
    """Fixed-size ring buffer for positions to avoid repeated allocations."""
    def __init__(self, seq_len):
        self.seq_len = seq_len
        self.buf = np.zeros((seq_len, 2), dtype=np.float32)
        self.ptr = 0
        self.count = 0  # how many valid entries (<= seq_len)

    def append(self, pos):
        # pos is (x,y)
        self.buf[self.ptr] = pos
        self.ptr = (self.ptr + 1) % self.seq_len
        if self.count < self.seq_len:
            self.count += 1

    def is_full(self):
        return self.count == self.seq_len

    def get_ordered(self):
        # return as (seq_len, 2) array oldest->newest (if not full, return available in order)
        if self.count == 0:
            return np.zeros((0, 2), dtype=np.float32)
        if self.count < self.seq_len:
            return self.buf[:self.count].copy()
        # full: pointer points to next write location => oldest is ptr
        idx = self.ptr
        return np.concatenate((self.buf[idx:], self.buf[:idx]), axis=0).copy()

# ---------------- Drawing helpers (lighter) ----------------
def draw_prediction_path(frame, current_pos, predicted_points, color):
    if predicted_points is None or len(predicted_points) == 0:
        return
    pts = [ (int(current_pos[0]), int(current_pos[1])) ]
    for p in predicted_points:
        pts.append((int(p[0]), int(p[1])))
    # Draw lines
    for i in range(len(pts)-1):
        cv2.line(frame, pts[i], pts[i+1], color, PREDICTION_THICKNESS)
    # Dots
    for p in pts:
        cv2.circle(frame, p, DOT_RADIUS, color, -1)

def draw_history_path(frame, history_points, color):
    if len(history_points) < 2:
        return
    pts = np.array(history_points, dtype=np.int32).reshape((-1,1,2))
    cv2.polylines(frame, [pts], isClosed=False, color=color, thickness=PATH_THICKNESS)
    cv2.circle(frame, tuple(map(int, history_points[-1])), DOT_RADIUS, color, -1)

# ---------------- Main script ----------------
print("--- Phase 1: Pre-Analysis of Bird Detections ---")
if not os.path.exists(DETECTIONS_CSV_PATH):
    raise FileNotFoundError(f"Detections CSV not found: {DETECTIONS_CSV_PATH}")

df_detections = pd.read_csv(DETECTIONS_CSV_PATH)
required_cols = {'frame','track_id','x1','y1','x2','y2'}
if not required_cols.issubset(set(df_detections.columns)):
    raise RuntimeError(f"Detections CSV missing required columns. Found: {df_detections.columns.tolist()}")

df_detections['center_x'] = (df_detections['x1'] + df_detections['x2']) / 2.0
df_detections['center_y'] = (df_detections['y1'] + df_detections['y2']) / 2.0

track_patterns, track_metrics = analyze_multi_bird_patterns(df_detections)

print(f"Identified {df_detections['track_id'].nunique()} unique bird tracks.")
for tid, metrics in track_metrics.items():
    pattern = track_patterns.get(tid, "N/A")
    print(f"Bird {tid}: {pattern} | Avg Vel: {metrics['avg_velocity']:.2f} px/frame | Frames: {metrics['frames_tracked']}")

print("\n--- Phase 2: Setup Video & TFLite ---")
detections_by_frame = {
    frame: {row['track_id']:{
        'position': np.array([row['center_x'], row['center_y']], dtype=np.float32),
        'bbox': np.array([row['x1'], row['y1'], row['x2'], row['y2']], dtype=np.int32)
    } for _, row in group.iterrows()}
    for frame, group in df_detections.groupby('frame')
}

# Load TFLite interpreter (try to load delegate if possible)
interpreter = None
try:
    # Attempt safe delegate load (most platforms have XNNPACK by default)
    try:
        delegate = None
        # Only attempt to load if experimental delegates available
        if hasattr(tf.lite.experimental, "load_delegate"):
            # No explicit path; let TF pick default delegate if available
            # NOTE: on some platforms this call can fail; catch exceptions
            delegate = tf.lite.experimental.load_delegate('libtensorflowlite_xnnpack_delegate.so')
            interpreter = tf.lite.Interpreter(model_path=TFLITE_MODEL_PATH, experimental_delegates=[delegate])
        else:
            interpreter = tf.lite.Interpreter(model_path=TFLITE_MODEL_PATH)
    except Exception:
        # Fallback to standard interpreter if delegate fails
        interpreter = tf.lite.Interpreter(model_path=TFLITE_MODEL_PATH)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    input_index = input_details[0]['index']
    output_index = output_details[0]['index']
    # pre-allocate input numpy buffer consistent with model shape
    input_shape = input_details[0]['shape']
    input_tensor_buffer = np.zeros(tuple(input_shape), dtype=np.float32)
    print("TFLite model loaded and tensors allocated.")
except Exception as e:
    raise RuntimeError(f"Failed to load TFLite model: {e}")

# Video I/O
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH}")

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) if cap.get(cv2.CAP_PROP_FPS) > 0 else 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (frame_width, frame_height))
print(f"Video opened: {frame_width}x{frame_height} @ {fps:.2f} FPS. Total frames: {total_frames}")

# ---------------- Processing variables ----------------
frame_number = 0
frame_times = []
prediction_times = []
track_response_times = {}

track_buffers = {}        # track_id -> TrackBuffer
track_predictions = {}    # track_id -> last predicted path (np.array shape (horizon,2))
track_colors = {}
track_first_seen = {}
color_palette = generate_colors(COLOR_PALETTE_SIZE)

# For display: rolling average compute at lower frequency
last_avg_pred_display = None
pred_display_counter = 0

print("\n--- Phase 3: Processing Video (optimized) ---")
while True:
    t0 = time.perf_counter()
    ret, frame = cap.read()
    if not ret:
        break

    current_frame_detections = detections_by_frame.get(frame_number, {})

    # Update each detected track
    for track_id, det in current_frame_detections.items():
        pos = det['position']
        bbox = det['bbox']

        if track_id not in track_first_seen:
            track_first_seen[track_id] = frame_number

        if track_id not in track_colors:
            track_colors[track_id] = color_palette[len(track_colors) % len(color_palette)]

        if track_id not in track_buffers:
            track_buffers[track_id] = TrackBuffer(SEQUENCE_LENGTH)

        # Append new position
        track_buffers[track_id].append(pos)

        # Draw bbox (light)
        c = track_colors[track_id]
        x1, y1, x2, y2 = int(bbox[0]), int(bbox[1]), int(bbox[2]), int(bbox[3])
        cv2.rectangle(frame, (x1, y1), (x2, y2), c, BOX_THICKNESS)

        # Decide whether to run prediction this frame for this track
        do_predict = False
        buf = track_buffers[track_id]
        if buf.is_full():
            # Use PRED_EVERY_N_FRAMES throttle to reduce inference frequency
            if (frame_number - track_first_seen[track_id]) % PRED_EVERY_N_FRAMES == 0:
                do_predict = True

        if do_predict:
            # Build ordered history oldest->newest
            ordered = buf.get_ordered()  # shape (SEQUENCE_LENGTH, 2)
            # Normalize
            normalized = ordered / np.array([frame_width, frame_height], dtype=np.float32)
            # Cast into preallocated input
            input_tensor_buffer[...] = 0.0
            # Try to handle shape (1, seq_len, 2) or more dims robustly
            try:
                input_tensor_buffer[0, :normalized.shape[0], :normalized.shape[1]] = normalized
            except Exception:
                # fallback: reshape directly
                input_tensor_buffer[:] = normalized.reshape(input_tensor_buffer.shape)

            # Run inference
            t_pred0 = time.perf_counter()
            interpreter.set_tensor(input_index, input_tensor_buffer.astype(np.float32))
            interpreter.invoke()
            pred_normalized = interpreter.get_tensor(output_index)  # shape depends on model
            t_pred1 = time.perf_counter()
            pred_ms = (t_pred1 - t_pred0) * 1000.0
            prediction_times.append(pred_ms)

            # Record response time first time
            if track_id not in track_response_times:
                track_response_times[track_id] = {
                    'detection_frame': track_first_seen[track_id],
                    'prediction_frame': frame_number,
                    'response_time_frames': frame_number - track_first_seen[track_id]
                }

            # Convert predicted normalized to pixel coords
            # Support models that output either flattened or (1,horizon,2)
            pred_arr = pred_normalized[0]
            if pred_arr.ndim == 1:
                # try to reshape if length matches horizon*2
                if pred_arr.size % 2 == 0:
                    pred_arr = pred_arr.reshape((-1, 2))
                else:
                    pred_arr = pred_arr.reshape((-1, 2))[:PREDICTION_HORIZON]
            pred_path_px = pred_arr * np.array([frame_width, frame_height], dtype=np.float32)
            track_predictions[track_id] = pred_path_px

    # Visualization: for tracks visible in this frame
    active_ids = set(current_frame_detections.keys())
    for track_id in active_ids:
        if track_id not in track_buffers:
            continue
        color = track_colors.get(track_id, (255,0,255))
        history = track_buffers[track_id].get_ordered()
        if history.shape[0] > 0:
            draw_history_path(frame, history.tolist(), color)
        if track_id in track_predictions:
            curr_pos = history[-1] if history.shape[0] > 0 else track_predictions[track_id][0]
            draw_prediction_path(frame, curr_pos, track_predictions[track_id], color)
            # Lightweight label
            pattern = track_patterns.get(track_id, "Unknown")
            label = f"ID:{track_id} {pattern}"
            lx = int(curr_pos[0]) + 6
            ly = int(curr_pos[1]) - 6
            # boundary check
            lx = min(max(0, lx), frame_width-1)
            ly = min(max(10, ly), frame_height-1)
            cv2.putText(frame, label, (lx, ly), FONT, FONT_SCALE, color, TEXT_THICKNESS, lineType=cv2.LINE_AA)

    # Draw summary performance text (update avg less frequently)
    if len(prediction_times) > 0:
        pred_display_counter += 1
        if pred_display_counter >= AVG_PRED_STATS_WINDOW:
            last_avg_pred_display = np.mean(prediction_times[-AVG_PRED_STATS_WINDOW:])
            pred_display_counter = 0
        if last_avg_pred_display is not None:
            cv2.putText(frame, f"Avg Pred: {last_avg_pred_display:.1f} ms", (10, 20), FONT, FONT_SCALE, (0,255,0), TEXT_THICKNESS)

    cv2.putText(frame, f"Frame: {frame_number}/{total_frames}", (10, 40), FONT, FONT_SCALE, (0,255,0), TEXT_THICKNESS)
    cv2.putText(frame, f"Active: {len(active_ids)}", (10, 60), FONT, FONT_SCALE, (0,255,0), TEXT_THICKNESS)

    out.write(frame)

    t1 = time.perf_counter()
    frame_time_ms = (t1 - t0) * 1000.0
    frame_times.append(frame_time_ms)

    if frame_number > 0 and frame_number % 100 == 0:
        avg_last = np.mean(frame_times[-100:]) if len(frame_times) >= 100 else np.mean(frame_times)
        print(f"Processed frame {frame_number}/{total_frames} - Avg Frame Time (last 100): {avg_last:.2f} ms")

    frame_number += 1

# ---------------- Finalize ----------------
cap.release()
out.release()
cv2.destroyAllWindows()

print("\n--- Phase 4: Finalizing and Report ---")
print(f"Saved video: {OUTPUT_VIDEO_PATH}")

if frame_times:
    print(f"Total Frames Processed: {len(frame_times)}")
    print(f"Average Frame Processing Time: {np.mean(frame_times):.2f} ms")
    print(f"Maximum Frame Processing Time: {np.max(frame_times):.2f} ms")
    print(f"Minimum Frame Processing Time: {np.min(frame_times):.2f} ms")
    target_frame_time = 1000.0 / fps if fps > 0 else 33.33
    real_time_factor = target_frame_time / np.mean(frame_times) if np.mean(frame_times) > 0 else 0.0
    print(f"Real-time Performance Factor: {real_time_factor:.2f} (>1.0 means real-time capable)")
    if real_time_factor >= 1.0:
        print("🟢 REAL-TIME CAPABLE")
    else:
        print("🔴 NOT REAL-TIME")

if prediction_times:
    print("\nTFLite predictions made:", len(prediction_times))
    print(f"Avg Prediction Time: {np.mean(prediction_times):.2f} ms")
    print(f"Max Prediction Time: {np.max(prediction_times):.2f} ms")
    if np.mean(prediction_times) > PREDICTION_TIME_WARNING_THRESHOLD_MS:
        print("⚠️ Prediction times are high; consider smaller model / quantization / faster delegate")

if track_response_times:
    print("\n--- Bird Tracking Response Times ---")
    for tid, tinfo in track_response_times.items():
        pattern = track_patterns.get(tid, "Unknown")
        print(f"Bird {tid} ({pattern}): detected {tinfo['detection_frame']} -> first_pred {tinfo['prediction_frame']}, frames={tinfo['response_time_frames']}")
        if fps > 0:
            print(f"  Response time: {(tinfo['response_time_frames'] / fps)*1000.0:.2f} ms")

print("\n=== PROCESSING COMPLETE ===")


--- Phase 1: Pre-Analysis of Bird Detections ---
Identified 83 unique bird tracks.
Bird 1: Unknown | Avg Vel: 12.93 px/frame | Frames: 52
Bird 2: Unknown | Avg Vel: 10.37 px/frame | Frames: 105
Bird 3: Unknown | Avg Vel: 8.64 px/frame | Frames: 10
Bird 4: Unknown | Avg Vel: 11.64 px/frame | Frames: 62
Bird 5: Maneuvering | Avg Vel: 4.35 px/frame | Frames: 90
Bird 6: Unknown | Avg Vel: 5.27 px/frame | Frames: 165
Bird 7: Maneuvering | Avg Vel: 8.52 px/frame | Frames: 15
Bird 8: Maneuvering | Avg Vel: 12.20 px/frame | Frames: 24
Bird 9: Maneuvering | Avg Vel: 13.74 px/frame | Frames: 36
Bird 10: Unknown | Avg Vel: 6.85 px/frame | Frames: 13
Bird 11: Maneuvering | Avg Vel: 9.48 px/frame | Frames: 84
Bird 12: Maneuvering | Avg Vel: 10.07 px/frame | Frames: 51
Bird 13: Maneuvering | Avg Vel: 17.36 px/frame | Frames: 8
Bird 18: Unknown | Avg Vel: 14.50 px/frame | Frames: 112
Bird 21: Erratic flight | Avg Vel: 15.86 px/frame | Frames: 17
Bird 25: Maneuvering | Avg Vel: 7.31 px/frame | Frames:

Exception ignored in: <function Delegate.__del__ at 0x000001D72F89DDA0>
Traceback (most recent call last):
  File "c:\Users\visha\AppData\Local\Programs\Python\Python312\Lib\site-packages\tensorflow\lite\python\interpreter.py", line 109, in __del__
    if self._library is not None:
       ^^^^^^^^^^^^^
AttributeError: 'Delegate' object has no attribute '_library'


TFLite model loaded and tensors allocated.
Video opened: 2560x1440 @ 24.00 FPS. Total frames: 279

--- Phase 3: Processing Video (optimized) ---
Processed frame 100/279 - Avg Frame Time (last 100): 43.62 ms
Processed frame 200/279 - Avg Frame Time (last 100): 42.61 ms

--- Phase 4: Finalizing and Report ---
Saved video: D:/Project_P1/results/video_analysis/video_with_multi_bird_predictions_optimized.mp4
Total Frames Processed: 279
Average Frame Processing Time: 43.19 ms
Maximum Frame Processing Time: 118.53 ms
Minimum Frame Processing Time: 31.62 ms
Real-time Performance Factor: 0.96 (>1.0 means real-time capable)
🔴 NOT REAL-TIME

TFLite predictions made: 1174
Avg Prediction Time: 0.75 ms
Max Prediction Time: 2.52 ms

--- Bird Tracking Response Times ---
Bird 1.0 (Unknown): detected 0 -> first_pred 14, frames=14
  Response time: 583.33 ms
Bird 5.0 (Maneuvering): detected 0 -> first_pred 14, frames=14
  Response time: 583.33 ms
Bird 6.0 (Unknown): detected 0 -> first_pred 14, frames=14


In [1]:
import os
import cv2
import numpy as np
import tensorflow as tf
from ultralytics import YOLO
from collections import defaultdict
import random
import time
import argparse

# =============================================================================
# --- Configuration & Parameters ---
# =============================================================================

# OPTIMIZATION 1: Frame Skipping
# Process the full YOLOv8 model only once every N frames.
# A value of 3 means we run the tracker on frame 1, use its results for frames 2 & 3,
# then run it again on frame 4, and so on. This can triple your FPS.
# Start with 2 or 3 and adjust as needed.
PROCESS_EVERY_N_FRAMES = 3

# Model & Sequence Parameters
SEQUENCE_LENGTH = 15
PREDICTION_HORIZON = 5
PRED_EVERY_N_FRAMES = 2 # This is for the TFLite model, can remain low

# Visualization Parameters
FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.5
BOX_THICKNESS = 1
PREDICTION_THICKNESS = 2
DOT_RADIUS = 3
TEXT_THICKNESS = 1
COLOR_PALETTE_SIZE = 50


# =============================================================================
# --- Helper Class: TrackBuffer (No changes needed) ---
# =============================================================================
class TrackBuffer:
    """A fixed-size ring buffer to efficiently store position history for each track."""
    def __init__(self, seq_len):
        self.seq_len = seq_len
        self.buf = np.zeros((seq_len, 2), dtype=np.float32)
        self.ptr = 0
        self.count = 0

    def append(self, pos):
        self.buf[self.ptr] = pos
        self.ptr = (self.ptr + 1) % self.seq_len
        if self.count < self.seq_len:
            self.count += 1

    def is_full(self):
        return self.count == self.seq_len

    def get_ordered(self):
        if self.count < self.seq_len:
            return self.buf[:self.count].copy()
        return np.roll(self.buf, -self.ptr, axis=0)


# =============================================================================
# --- Main Class: RealTimeBirdPredictor ---
# =============================================================================
class RealTimeBirdPredictor:
    """
    Integrates YOLO tracking and TFLite prediction for real-time video processing.
    """
    def __init__(self, yolo_path, tflite_path, conf_threshold=0.3):
        print("Initializing the real-time bird prediction system... 🚀")
        self.yolo_model = YOLO(yolo_path)
        print("✅ YOLOv8 model loaded.")
        try:
            self.interpreter = tf.lite.Interpreter(model_path=tflite_path)
            self.interpreter.allocate_tensors()
            self.input_details = self.interpreter.get_input_details()
            self.output_details = self.interpreter.get_output_details()
            self.input_index = self.input_details[0]['index']
            self.output_index = self.output_details[0]['index']
            print("✅ TFLite LSTM model loaded.")
        except Exception as e:
            raise ValueError(f"Error loading TFLite model: {e}")

        self.conf_threshold = conf_threshold
        self.track_buffers = defaultdict(lambda: TrackBuffer(SEQUENCE_LENGTH))
        self.track_predictions = {}
        self.track_colors = {}
        self.track_first_seen = {}
        self.color_palette = self._generate_colors(COLOR_PALETTE_SIZE)
        self.frame_times = []
        self.prediction_times = []

    def _generate_colors(self, num_colors):
        rng = random.Random(42)
        colors = []
        for _ in range(num_colors):
            colors.append(tuple(rng.randint(60, 255) for _ in range(3)))
        return colors

    def _draw_prediction(self, frame, current_pos, predicted_points, color):
        if predicted_points is None or len(predicted_points) == 0:
            return
        all_points = [tuple(map(int, current_pos))] + [tuple(map(int, p)) for p in predicted_points]
        for i in range(len(all_points) - 1):
            cv2.line(frame, all_points[i], all_points[i+1], color, PREDICTION_THICKNESS)
        cv2.circle(frame, all_points[0], DOT_RADIUS + 1, color, -1)

    def process_stream(self, source, output_path=None):
        is_webcam = str(source).lower() == 'webcam'
        cap_source = 0 if is_webcam else source
        cap = cv2.VideoCapture(cap_source)
        if not cap.isOpened():
            raise IOError(f"Cannot open video source: {source}")

        frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS) if not is_webcam else 30
        out = None
        if output_path:
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))
            print(f"🎥 Output will be saved to: {output_path}")

        frame_number = 0
        last_results = None # Store the results from the last processed frame

        while True:
            t_start = time.perf_counter()
            ret, frame = cap.read()
            if not ret:
                print("End of video stream.")
                break

            active_track_ids = set()

            # --- Core Optimization: Process frames intermittently ---
            if frame_number % PROCESS_EVERY_N_FRAMES == 0:
                # Run the expensive tracker
                results = self.yolo_model.track(
                    frame,
                    persist=True,
                    conf=self.conf_threshold,
                    imgsz=640, # OPTIMIZATION 2: Reduce processing resolution
                    verbose=False
                )
                last_results = results[0] # Cache the results

                # Process detections and run predictions ONLY when we get new tracking data
                if last_results is not None and last_results.boxes is not None and last_results.boxes.id is not None:
                    track_ids = last_results.boxes.id.int().cpu().tolist()
                    boxes = last_results.boxes.xyxy.cpu()
                    active_track_ids.update(track_ids)

                    for i, track_id in enumerate(track_ids):
                        box = boxes[i]
                        center_x = (box[0] + box[2]) / 2.0
                        center_y = (box[1] + box[3]) / 2.0
                        current_position = np.array([center_x, center_y], dtype=np.float32)

                        if track_id not in self.track_first_seen:
                            self.track_first_seen[track_id] = frame_number
                        self.track_buffers[track_id].append(current_position)

                        buffer = self.track_buffers[track_id]
                        if buffer.is_full() and (frame_number - self.track_first_seen[track_id]) % PRED_EVERY_N_FRAMES == 0:
                            history = buffer.get_ordered()
                            normalized_history = history / np.array([frame_width, frame_height], dtype=np.float32)
                            input_tensor = np.expand_dims(normalized_history, axis=0).astype(np.float32)
                            
                            t_pred_start = time.perf_counter()
                            self.interpreter.set_tensor(self.input_index, input_tensor)
                            self.interpreter.invoke()
                            pred_normalized = self.interpreter.get_tensor(self.output_index)
                            self.prediction_times.append((time.perf_counter() - t_pred_start) * 1000)

                            pred_path = pred_normalized[0] * np.array([frame_width, frame_height], dtype=np.float32)
                            self.track_predictions[track_id] = pred_path
            
            # --- Visualization (runs on EVERY frame for smoothness) ---
            # We use the 'last_results' so skipped frames still show boxes
            if last_results is not None and last_results.boxes is not None and last_results.boxes.id is not None:
                track_ids_to_draw = last_results.boxes.id.int().cpu().tolist()
                boxes_to_draw = last_results.boxes.xyxy.cpu()

                for i, track_id in enumerate(track_ids_to_draw):
                    # Draw bounding box from last known result
                    box = boxes_to_draw[i]
                    color = self.track_colors.setdefault(track_id, self.color_palette[len(self.track_colors) % COLOR_PALETTE_SIZE])
                    cv2.rectangle(frame, (int(box[0]), int(box[1])), (int(box[2]), int(box[3])), color, BOX_THICKNESS)
                    
                    # Draw prediction and label
                    if track_id in self.track_predictions:
                        history = self.track_buffers[track_id].get_ordered()
                        if history.shape[0] > 0:
                            self._draw_prediction(frame, history[-1], self.track_predictions[track_id], color)
                            last_pos = history[-1]
                            label = f"ID:{track_id}"
                            cv2.putText(frame, label, (int(last_pos[0]) + 7, int(last_pos[1]) - 7), FONT, FONT_SCALE, color, TEXT_THICKNESS)
            
            # Display Performance Metrics
            t_end = time.perf_counter()
            frame_time_ms = (t_end - t_start) * 1000
            self.frame_times.append(frame_time_ms)
            avg_frame_time = np.mean(self.frame_times[-30:]) if self.frame_times else 0
            avg_pred_time = np.mean(self.prediction_times[-30:]) if self.prediction_times else 0
            
            # Calculate FPS based on the rolling average for a smoother display
            display_fps = 1000.0 / avg_frame_time if avg_frame_time > 0 else 0

            cv2.putText(frame, f"Frame Time: {avg_frame_time:.1f} ms (FPS: {display_fps:.1f})", (10, 20), FONT, FONT_SCALE, (0, 255, 0), TEXT_THICKNESS)
            cv2.putText(frame, f"Prediction Time: {avg_pred_time:.1f} ms", (10, 40), FONT, FONT_SCALE, (0, 255, 0), TEXT_THICKNESS)
            
            # Show and Save Frame
            cv2.imshow("Real-Time Bird Prediction", frame)
            if out:
                out.write(frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                print("Interrupted by user.")
                break
            
            frame_number += 1
            
        cap.release()
        if out:
            out.release()
        cv2.destroyAllWindows()
        print("\n--- Processing Complete ---")
        if self.frame_times:
            avg_total_fps = 1000 / np.mean(self.frame_times)
            print(f"Average processing FPS: {avg_total_fps:.2f}")
            print(f"Average frame processing time: {np.mean(self.frame_times):.2f} ms")
        if self.prediction_times:
            print(f"Average TFLite prediction time: {np.mean(self.prediction_times):.2f} ms")

# =============================================================================
# --- Script Entry Point (Modified for Notebook) ---
# =============================================================================
if __name__ == "__main__":
    # --- Hardcode your paths and settings here ---
    SOURCE_PATH = "D:/Project_P1/data/valid/test1.mp4" 
    
    YOLO_MODEL_PATH = "D:/Project_P1/src/ml/runs/detect/train2/weights/best.pt" # Your original model
    
    TFLITE_MODEL_PATH = "D:/Project_P1/results/video_analysis/model.tflite"
    OUTPUT_FILE_PATH = "D:/Project_P1/results/optimized_output.mp4"
    CONFIDENCE_THRESHOLD = 0.3
    # ---------------------------------------------

    try:
        predictor = RealTimeBirdPredictor(
            yolo_path=YOLO_MODEL_PATH,
            tflite_path=TFLITE_MODEL_PATH,
            conf_threshold=CONFIDENCE_THRESHOLD
        )
        predictor.process_stream(
            source=SOURCE_PATH,
            output_path=OUTPUT_FILE_PATH
        )
    except Exception as e:
        print(f"An error occurred: {e}")

Initializing the real-time bird prediction system... 🚀
✅ YOLOv8 model loaded.
✅ TFLite LSTM model loaded.
🎥 Output will be saved to: D:/Project_P1/results/optimized_output.mp4
End of video stream.

--- Processing Complete ---
Average processing FPS: 12.79
Average frame processing time: 78.20 ms
Average TFLite prediction time: 1.04 ms


In [1]:
from ultralytics import YOLO

# Load your trained YOLOv8 model
# Replace 'yolov8n.pt' with the path to your own model if needed
model = YOLO('D:/Project_P1/src/ml/runs/detect/train2/weights/best.pt')

# Perform detection on an image
# Replace with the path to your image
results = model.predict('D:/Project_P1/data/valid/images/1005_jpg.rf.54152cb974aeb70afafbb0321922d420.jpg')

# The 'speed' attribute contains the timings in milliseconds (ms)
# It's a dictionary with keys: 'preprocess', 'inference', 'postprocess'
speed_ms = results[0].speed

print(f"Pre-processing time: {speed_ms['preprocess']:.2f} ms")
print(f"✅ Inference (detection) time: {speed_ms['inference']:.2f} ms")
print(f"Post-processing time: {speed_ms['postprocess']:.2f} ms")

WARNING torchvision==0.22 is incompatible with torch==2.3.
Run 'pip install torchvision==0.18' to fix torchvision or 'pip install -U torch torchvision' to update both.
For a full compatibility table see https://github.com/pytorch/vision#installation

image 1/1 D:\Project_P1\data\valid\images\1005_jpg.rf.54152cb974aeb70afafbb0321922d420.jpg: 384x640 1 1, 236.0ms
Speed: 9.4ms preprocess, 236.0ms inference, 28.2ms postprocess per image at shape (1, 3, 384, 640)
Pre-processing time: 9.36 ms
✅ Inference (detection) time: 236.02 ms
Post-processing time: 28.17 ms


In [3]:
import torch
from ultralytics import YOLO

# Load your trained YOLOv8 model
model = YOLO('D:/Project_P1/src/ml/runs/detect/train2/weights/best.pt')

# --- ADD THIS SECTION FOR GPU SUPPORT ---
# Check if a CUDA-enabled GPU is available and move the model to it
if torch.cuda.is_available():
    model.to('cuda')
    print("✅ Model moved to GPU for faster inference.")
else:
    print("⚠️ GPU not found, using CPU for inference.")
# -----------------------------------------

# Perform detection on an image
results = model.predict('D:/Project_P1/data/valid/images/1005_jpg.rf.54152cb974aeb70afafbb0321922d420.jpg')

# The rest of your code remains the same
speed_ms = results[0].speed

print(f"\nPre-processing time: {speed_ms['preprocess']:.2f} ms")
print(f"Inference (detection) time: {speed_ms['inference']:.2f} ms")
print(f"Post-processing time: {speed_ms['postprocess']:.2f} ms")

✅ Model moved to GPU for faster inference.

image 1/1 D:\Project_P1\data\valid\images\1005_jpg.rf.54152cb974aeb70afafbb0321922d420.jpg: 384x640 1 1, 20.8ms
Speed: 4.1ms preprocess, 20.8ms inference, 3.3ms postprocess per image at shape (1, 3, 384, 640)

Pre-processing time: 4.14 ms
Inference (detection) time: 20.81 ms
Post-processing time: 3.26 ms


In [1]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)


CUDA available: True
Device: NVIDIA GeForce RTX 2050
CUDA version: 12.4


### Review - 03

In [1]:
import os
import cv2
import pandas as pd
import numpy as np
import tensorflow as tf
from collections import defaultdict
import random
import time
import math

# ---------------- Configuration ----------------
DETECTIONS_CSV_PATH = "D:/Project_P1/results/video_analysis/test1_detections_with_ids.csv"
VIDEO_PATH = "D:/Project_P1/data/valid/test1.mp4"
TFLITE_MODEL_PATH = "D:/Project_P1/results/video_analysis/model.tflite"
OUTPUT_VIDEO_PATH = "D:/Project_P1/results/video_analysis/video_with_multi_bird_predictions_optimized.mp4"

SEQUENCE_LENGTH = 15
PREDICTION_HORIZON = 5
PRED_EVERY_N_FRAMES = 2

MAX_PROCESSING_TIME_MS = 33
PREDICTION_TIME_WARNING_THRESHOLD_MS = 10

FONT = cv2.FONT_HERSHEY_SIMPLEX
FONT_SCALE = 0.45
PATH_THICKNESS = 1
BOX_THICKNESS = 1
PREDICTION_THICKNESS = 1
DOT_RADIUS = 3
TEXT_THICKNESS = 1

COLOR_PALETTE_SIZE = 50
AVG_PRED_STATS_WINDOW = 30

# ---------------- Utility ----------------
def generate_colors(num_colors):
    colors = []
    rng = random.Random(42)
    for i in range(num_colors):
        r = rng.randint(60, 255)
        g = rng.randint(60, 255)
        b = rng.randint(60, 255)
        colors.append((int(b), int(g), int(r)))
    return colors

# ---------------- Flight Pattern Analysis ----------------
def analyze_multi_bird_patterns(df):
    patterns = {}
    metrics = {}
    for track_id, group in df.groupby('track_id'):
        if len(group) < 5:
            continue
        seq_data = group.sort_values('frame').copy()
        x = seq_data['center_x'].values
        y = seq_data['center_y'].values
        dx, dy = np.gradient(x), np.gradient(y)
        velocity = np.sqrt(dx**2 + dy**2)
        ddx, ddy = np.gradient(dx), np.gradient(dy)
        acceleration = np.sqrt(ddx**2 + ddy**2)
        dddx, dddy = np.gradient(ddx), np.gradient(ddy)
        jerk = np.sqrt(dddx**2 + dddy**2)
        directions = np.arctan2(dy, dx)
        turning_angles = np.abs(np.diff(directions))
        turning_angles = np.append(turning_angles, 0)
        avg_vel, std_vel = np.mean(velocity), np.std(velocity)
        avg_acc, std_acc = np.mean(acceleration), np.std(acceleration)
        avg_jerk = np.mean(jerk)
        avg_turn = np.mean(turning_angles)
        pattern_type = "Unknown"
        if std_vel < 5 and avg_acc < 2 and avg_turn < 0.2:
            pattern_type = "Steady flight"
        elif avg_acc > 5 or avg_turn > 0.5:
            pattern_type = "Maneuvering"
        elif std_vel > 10 or avg_jerk > 3:
            pattern_type = "Erratic flight"
        patterns[track_id] = pattern_type
        metrics[track_id] = {
            'avg_velocity': avg_vel,
            'std_velocity': std_vel,
            'avg_acceleration': avg_acc,
            'std_acceleration': std_acc,
            'avg_jerk': avg_jerk,
            'avg_turning_angle': avg_turn,
            'frames_tracked': len(group)
        }
    return patterns, metrics

# ---------------- Per-track ring buffer ----------------
class TrackBuffer:
    def __init__(self, seq_len):
        self.seq_len = seq_len
        self.buf = np.zeros((seq_len, 2), dtype=np.float32)
        self.ptr = 0
        self.count = 0

    def append(self, pos):
        self.buf[self.ptr] = pos
        self.ptr = (self.ptr + 1) % self.seq_len
        if self.count < self.seq_len:
            self.count += 1

    def is_full(self):
        return self.count == self.seq_len

    def get_ordered(self):
        if self.count == 0:
            return np.zeros((0, 2), dtype=np.float32)
        if self.count < self.seq_len:
            return self.buf[:self.count].copy()
        idx = self.ptr
        return np.concatenate((self.buf[idx:], self.buf[:idx]), axis=0).copy()

# ---------------- Drawing helpers ----------------
def draw_prediction_path(frame, current_pos, predicted_points, color):
    if predicted_points is None or len(predicted_points) == 0:
        return
    pts = [(int(current_pos[0]), int(current_pos[1]))]
    for p in predicted_points:
        pts.append((int(p[0]), int(p[1])))
    for i in range(len(pts)-1):
        cv2.line(frame, pts[i], pts[i+1], color, PREDICTION_THICKNESS)
    for p in pts:
        cv2.circle(frame, p, DOT_RADIUS, color, -1)

def draw_history_path(frame, history_points, color):
    if len(history_points) < 2:
        return
    pts = np.array(history_points, dtype=np.int32).reshape((-1,1,2))
    cv2.polylines(frame, [pts], isClosed=False, color=color, thickness=PATH_THICKNESS)
    cv2.circle(frame, tuple(map(int, history_points[-1])), DOT_RADIUS, color, -1)

# ---------------- Main ----------------
print("--- Phase 1: Load Data ---")
if not os.path.exists(DETECTIONS_CSV_PATH):
    raise FileNotFoundError(f"Detections CSV not found: {DETECTIONS_CSV_PATH}")

df_detections = pd.read_csv(DETECTIONS_CSV_PATH)
df_detections['center_x'] = (df_detections['x1'] + df_detections['x2']) / 2.0
df_detections['center_y'] = (df_detections['y1'] + df_detections['y2']) / 2.0

track_patterns, track_metrics = analyze_multi_bird_patterns(df_detections)

# --- Video Setup ---
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH}")

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS) if cap.get(cv2.CAP_PROP_FPS) > 0 else 30.0
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (frame_width, frame_height))
print(f"Video: {frame_width}x{frame_height} @ {fps:.2f} FPS, {total_frames} frames")

# --- Load TFLite ---
interpreter = tf.lite.Interpreter(model_path=TFLITE_MODEL_PATH)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
input_index = input_details[0]['index']
output_index = output_details[0]['index']
input_shape = input_details[0]['shape']
input_tensor_buffer = np.zeros(tuple(input_shape), dtype=np.float32)

detections_by_frame = {
    frame: {row['track_id']:{
        'position': np.array([row['center_x'], row['center_y']], dtype=np.float32),
        'bbox': np.array([row['x1'], row['y1'], row['x2'], row['y2']], dtype=np.int32)
    } for _, row in group.iterrows()}
    for frame, group in df_detections.groupby('frame')
}

# --- Tracking setup ---
frame_times, prediction_times, frame_draw_times = [], [], []
track_buffers, track_predictions, track_colors, track_first_seen = {}, {}, {}, {}
color_palette = generate_colors(COLOR_PALETTE_SIZE)
frame_number = 0

print("\n--- Phase 2: Processing Video ---")
while True:
    t0 = time.perf_counter()
    ret, frame = cap.read()
    if not ret:
        break

    current_frame_detections = detections_by_frame.get(frame_number, {})

    for track_id, det in current_frame_detections.items():
        pos = det['position']
        bbox = det['bbox']

        if track_id not in track_first_seen:
            track_first_seen[track_id] = frame_number
        if track_id not in track_colors:
            track_colors[track_id] = color_palette[len(track_colors) % len(color_palette)]
        if track_id not in track_buffers:
            track_buffers[track_id] = TrackBuffer(SEQUENCE_LENGTH)

        track_buffers[track_id].append(pos)
        c = track_colors[track_id]
        cv2.rectangle(frame, (int(bbox[0]), int(bbox[1])), (int(bbox[2]), int(bbox[3])), c, BOX_THICKNESS)

        do_predict = False
        buf = track_buffers[track_id]
        if buf.is_full() and (frame_number - track_first_seen[track_id]) % PRED_EVERY_N_FRAMES == 0:
            do_predict = True

        if do_predict:
            ordered = buf.get_ordered()
            normalized = ordered / np.array([frame_width, frame_height], dtype=np.float32)
            input_tensor_buffer[...] = 0.0
            input_tensor_buffer[0, :normalized.shape[0], :normalized.shape[1]] = normalized

            t_pred0 = time.perf_counter()
            interpreter.set_tensor(input_index, input_tensor_buffer.astype(np.float32))
            interpreter.invoke()
            pred_normalized = interpreter.get_tensor(output_index)
            t_pred1 = time.perf_counter()
            pred_ms = (t_pred1 - t_pred0) * 1000.0
            prediction_times.append(pred_ms)

            pred_arr = pred_normalized[0]
            if pred_arr.ndim == 1 and pred_arr.size % 2 == 0:
                pred_arr = pred_arr.reshape((-1, 2))
            pred_path_px = pred_arr * np.array([frame_width, frame_height], dtype=np.float32)
            track_predictions[track_id] = pred_path_px

    # --- Measure only OpenCV drawing ---
    t_draw0 = time.perf_counter()

    active_ids = set(current_frame_detections.keys())
    for track_id in active_ids:
        color = track_colors.get(track_id, (255,0,255))
        history = track_buffers[track_id].get_ordered()
        if history.shape[0] > 0:
            draw_history_path(frame, history.tolist(), color)
        if track_id in track_predictions:
            curr_pos = history[-1]
            draw_prediction_path(frame, curr_pos, track_predictions[track_id], color)
            pattern = track_patterns.get(track_id, "Unknown")
            cv2.putText(frame, f"ID:{track_id} {pattern}", (int(curr_pos[0])+6, int(curr_pos[1])-6),
                        FONT, FONT_SCALE, color, TEXT_THICKNESS, lineType=cv2.LINE_AA)

    cv2.putText(frame, f"Frame: {frame_number}/{total_frames}", (10, 20), FONT, FONT_SCALE, (0,255,0), TEXT_THICKNESS)
    cv2.putText(frame, f"Active: {len(active_ids)}", (10, 40), FONT, FONT_SCALE, (0,255,0), TEXT_THICKNESS)

    out.write(frame)

    t_draw1 = time.perf_counter()
    frame_draw_times.append((t_draw1 - t_draw0) * 1000.0)

    t1 = time.perf_counter()
    frame_times.append((t1 - t0) * 1000.0)

    frame_number += 1

cap.release()
out.release()
cv2.destroyAllWindows()

# ---------------- Final Summary ----------------
print("\n--- Final Performance Summary ---")

yolo_time = 29.0  # measured earlier
lstm_time = np.mean(prediction_times) if prediction_times else 1.18
opencv_time = np.mean(frame_draw_times) if frame_draw_times else 0.0

print(f"{'Component':<30} | {'Time per Frame (ms)':>20} | {'Total (390 frames)':>20}")
print("-" * 75)
print(f"{'YOLOv8 Detection':<30} | {yolo_time:>20.2f} | {yolo_time*390/1000:>19.2f}s")
print(f"{'LSTM (TFLite) Prediction':<30} | {lstm_time:>20.2f} | {lstm_time*390/1000:>19.2f}s")
print(f"{'OpenCV Plotting & Write':<30} | {opencv_time:>20.2f} | {opencv_time*390/1000:>19.2f}s")
print("-" * 75)
total_per_frame = yolo_time + lstm_time + opencv_time
print(f"{'TOTAL':<30} | {total_per_frame:>20.2f} | {total_per_frame*390/1000:>19.2f}s")
print("\n✅ Saved:", OUTPUT_VIDEO_PATH)


--- Phase 1: Load Data ---
Video: 2560x1440 @ 24.00 FPS, 279 frames

--- Phase 2: Processing Video ---

--- Final Performance Summary ---
Component                      |  Time per Frame (ms) |   Total (390 frames)
---------------------------------------------------------------------------
YOLOv8 Detection               |                29.00 |               11.31s
LSTM (TFLite) Prediction       |                 0.93 |                0.36s
OpenCV Plotting & Write        |                29.88 |               11.65s
---------------------------------------------------------------------------
TOTAL                          |                59.81 |               23.33s

✅ Saved: D:/Project_P1/results/video_analysis/video_with_multi_bird_predictions_optimized.mp4
